In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget
from matplotlib.colors import Normalize

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

import scipy.stats as stats
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 100)     # Show all rows
pd.set_option('display.width', 500)        # Wide display
pd.set_option('display.max_colwidth', 40) # Full width column content
pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple lines

In [ ]:
# for poster, use larger fonts

plt.rcParams.update({
    'font.size': 15,           # general font size
    'axes.titlesize': 15,      # subplot title
    'axes.labelsize': 15,      # x and y labels
    'xtick.labelsize': 15,     # x tick labels
    'ytick.labelsize': 15,     # y tick labels
    'legend.fontsize': 15,     # legend
})

In [ ]:
# first set the paths and logger. You can see all kind of outputs in this directory 
# already. Poster used plots from this directory.

data = {}
nas_dir = C.device_paths()[0]
output_dir = f"{nas_dir}/Simon/poster/cue_decoding/"
Logger().init_logger(None, None, logging_level="WARNING")

In [ ]:
# analysis was intially done on all sessions from animal 6 with paradigm 1100
# But the poster focused on S8-15

animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [ ]:
# this is a new analytic. It is a crucial component that is required by the SVM
# decoding analytic to be computed. It is also used here for plotting.
# Another input analytic that is used to calc TrialWiseT0Events40ms that I use 
# now a lot is Behavior40msAligned - this is similar to BehaviorTrackwise, but 
# instead of binning according to track position, we bin according to the 40ms 
# ephys bins. So usually an average over 2 or 3 unity frames.

# Each row is an interval - a start and a stop ephys time. Think of this as the 
# selecter of firingrates X for the SVM that we want to train with. There are 4 
# differnent intervals a row may define (and 2 more specifcal-case ones): 
# cue_entry_interval, cue_exit_interval, R1_entry_interval, R2_entry_interval
# (cue_exit_interval is not used, in reality it overlaps a lot with R1_entry_interval).
# 
# So in general, each row is an interval of one of these kinds. All other intervals 
# are NaN in that row (so each interval has it's own, sparse column)
# Columns: trial_id	cue	trial_outcome	choice_R1	choice_R2	t0_event_name	t0	x_position	x_alignment	pre_cue_interval	nextto_cue_interval	cue_entry_interval	R1_entry_interval	R2_entry_interval
# So you can see we have the labels, a name of the interval (t0_event_name), the t0 time, and importantly, the x_position 
# we tried to align to (x_alignment) and finally the actual x_position that the 
# unty frame had. So you can see the way it works is for every trial, we check a 
# set of x_alignments (e.g. cue position, R1 position, R2 position) and find the
# ephys time (t0) that corresponds to that x_position. Then we define intervals
# around that t0. The length of the intervals can differ. 

# There are also two special intervals, pre_cue_interval, and nextto_cue_interval 
# which are in the same row and short, 200ms,  or 5 ephys bins. The ones above have 
# usually around 30 ephys bins. These were used to train an SVM to decode pre- vs 
# within cue neural activity, first for cue1 and then for cue2 trials. Then I 
# tried to compare these two fits to check if they are orthogonal.

# all of this is calcuclated in analytics/integr_analytics.py

t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events


In [ ]:
# this is the new analyric I metioned Behavior40msAligned. I think we just need this
# to draw x positions at every timepoint in one of the plots

behavior = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
behavior.iloc[100:200]


In [ ]:
behavior.set_index('to_ephys_timestamp', append=True, inplace=True)
behavior.index = behavior.index.droplevel('entry_id')
behavior

In [ ]:
# get the raw firing rates, i don't rmember why

fr = fr.set_index(["from_ephys_timestamp","to_ephys_timestamp"], append=True)
fr.reset_index(level=(0,1,3,4), inplace=True, drop=True)
fr

In [ ]:
behavior

In [ ]:
# confirm that t- events are in the right region using bodycam

# camera = 'bodycam'
# plt.close("all")
# for s_id in t0_events.index.unique('session_id'):
#     if s_id <8:
#         continue
    
#     data = session_modality_from_nas(session_dirs[s_id], f"{camera}_packages")
#     for t0 in t0_events.loc[s_id].index[::19]:
#         print(t0_events.loc[s_id, t0])
        
#         fig, axes = plt.subplots(ncols=6, figsize=(20,3))
#         fig.suptitle(f'Session {s_id}, t0: {t0}', fontsize=16)
#         for i, delta in enumerate(range(-12, 12, 4)):
#             closest_frame = data.iloc[(data[f'{camera}_image_ephys_timestamp'] - t0).abs().argsort()[0] +delta]
#             frame_id = int(closest_frame.loc[f'{camera}_image_id']) 
#             ephys_t = closest_frame.loc[f'{camera}_image_ephys_timestamp'] + (delta * 40000)
            
#             closest_beh_t = (behavior.xs(s_id, level='session_id').loc[:, 'from_ephys_timestamp'] - ephys_t).abs().argsort().values[0]
#             closest_beh_frame = behavior.xs(s_id, level='session_id').iloc[closest_beh_t]
            
#             # print(f"Frame id: {frame_id}, ephys time: {ephys_t}, closest behavior time: {closest_beh_t}, frame position: {closest_beh_frame['frame_position']}")
#             # print(f"Looking for t0: {t0}, closest frame id: {frame_id}, ephys time: {ephys_t}, diff: {ephys_t - t0}")
#             # print(f"Position at t0: {t0_events.loc[(s_id, t0), 'x_position']}")
#             title = f'x: {closest_beh_frame['frame_position']:.2f}cm, \nt_diff: {(ephys_t - t0)/1e6:.3f} us, \ndelta={delta}'
            
#             frame_key = f'frame_{frame_id:06d}'

#             img = session_modality_from_nas(session_dirs[s_id], key=f"{camera}_frames", columns=[frame_key])
#             axes[i].imshow(img[0])
#             axes[i].axis('off')
#             axes[i].set_title(title, fontsize=8)
        
#         plt.tight_layout()
#         plt.show()
#         # break
#     break
        

In [ ]:
# the main data to plot. Calculated like 1000 other things in postproc_mea1k_ephys.py....
# columns w	predictions	n_positives	n_negatives	f1_aggr	f1_mean	f1_ci_lower	f1_ci_upper	acc_mean	acc_ci_lower	acc_ci_upper	X_modality	predict_y_name	interval_name	timebin	y_true
# so we have the weight vecter (avg over all bootstraps), the predictions, the number of positive and negative labels, 
# a bunch of performance metrics, X_madilty = the input data used (X), 
# predict_y_name the SVM that was fit - this is closely linked to the interval used.
# interval_name, the interval name used to select X
# timebin, very important, the specfic timebin used for fitting. This goes from 0 to eg 30. 
# Remeber that in one interval we do a fit for each timebin - we slide by one 
# timebin (40 ms ephys bin) The intervals from from T0Events analytic. This is 
# where we select one timepoint as t0 (eg cue visible) and say lets look at 10 
# timebins before that and 20 after that event. For each of those timepoints 
# we fit a SVM.

# in this case below, we slice to only the special case I described before: 
# pre vs within cue zone for cue1 and cue2 (didn't make it on to the poster, but
# I explained this to you before I remember) This one is not too important, don't
# need to focus on it

data = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
data = data[data.predict_y_name.isin(['pre_vs_in_cue1', 'pre_vs_in_cue2'])]
data

In [ ]:
# This is one of the first plots I made, it's not in the output NAS dir yet. Run it

plt.close("all")
x_modalities = ['HP+mPFC', 'mPFC', 'HP']

no_ephys_s_ids = [0,5,8,17,25,31,29]
ss_cropped_s_ids = [10,24]

# drop sessions with no ephys data
data = data[~data.index.get_level_values('session_id').isin(no_ephys_s_ids)]
data = data[~data.index.get_level_values('session_id').isin(ss_cropped_s_ids)]

for x_modality in x_modalities:
    subset = data[data['X_modality'] == x_modality]
    plt.figure(figsize=(10,4))
    
    # Define colors for each cue
    cue_colors = {'pre_vs_in_cue1': 'orange', 'pre_vs_in_cue2': 'purple'}
    
    for predict_y in ['pre_vs_in_cue1', 'pre_vs_in_cue2']:
        cue_data = subset[subset['predict_y_name'] == predict_y]
        plt.scatter(cue_data.index.unique('session_id'), 
                   cue_data['f1_mean'],
                   color=cue_colors[predict_y],
                   label=f'{predict_y}')
        # add error bars
        for s_id in cue_data.index.unique('session_id'):
             plt.plot([s_id, s_id], 
                      [cue_data.loc[(cue_data.index.get_level_values('session_id') == s_id), 'f1_ci_lower'].values[0],
                       cue_data.loc[(cue_data.index.get_level_values('session_id') == s_id), 'f1_ci_upper'].values[0]],
                      color=cue_colors[predict_y], alpha=0.5)

    # Add reference lines and markers
    plt.scatter(no_ephys_s_ids, [.15]*len(no_ephys_s_ids), color='gray', marker='x', s=50, label='No Ephys Sessions')
    plt.scatter(ss_cropped_s_ids, [.12]*len(ss_cropped_s_ids), color='lightgray', marker='x', s=50, label='SS cropped')
    plt.axvline(x=25.5, color='blue', alpha=.4, linestyle='--', label='Reversal after')
    plt.axvline(x=7.5, color='gray', linestyle='--', label='double rewards before')
    
    plt.title(f'SVM F1 Score over Time - {x_modality}')
    plt.axhline(.5, color='black', linestyle=':', label='Chance Level')
    plt.ylim(.1,.9)
    plt.xlabel('Session ID')
    plt.xticks(np.arange(34))
    plt.ylabel('Before vs within-cue zone SVM decoding F1')
    plt.tight_layout()
    plt.legend()
    plt.show()
    
    # Plot angles between SVM weights
    plt.figure(figsize=(10,4))
    for s_id in subset.index.unique('session_id'):
        cue1_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue1')]
        cue2_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue2')]
        
        if not cue1_data.empty and not cue2_data.empty:
            cue1_w = cue1_data['w'].values[0]
            cue2_w = cue2_data['w'].values[0]
            cos_angle = np.dot(cue1_w, cue2_w) / (np.linalg.norm(cue1_w) * np.linalg.norm(cue2_w))
            angle_deg = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * (180/np.pi)
            plt.scatter(s_id, angle_deg, color='teal')
            
    plt.title(f'Angle between Cue 1 and Cue 2 SVM Weights - {x_modality}')
    plt.axhline(90, color='black', linestyle=':', label='Orthogonal (90 degrees)')
    plt.xlabel('Session ID')
    plt.xticks(np.arange(34))
    plt.ylabel('Angle (degrees)')
    plt.ylim(0,180)
    plt.tight_layout()
    plt.show()

    # Correlation plots
    angles = []
    f1_scores = []
    for s_id in subset.index.unique('session_id'):
        cue1_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue1')]
        cue2_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue2')]
        
        if not cue1_data.empty and not cue2_data.empty:
            cue1_w = cue1_data['w'].values[0]
            cue2_w = cue2_data['w'].values[0]
            cos_angle = np.dot(cue1_w, cue2_w) / (np.linalg.norm(cue1_w) * np.linalg.norm(cue2_w))
            angle_deg = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * (180/np.pi)
            angles.append(angle_deg)
            f1_scores.append(cue1_data['f1_mean'].values[0])

    if angles and f1_scores:
        plt.figure(figsize=(6,4))
        plt.scatter(angles, f1_scores, color='brown')
        m, b = np.polyfit(angles, f1_scores, 1)
        plt.plot(angles, m*np.array(angles) + b, color='orange', linestyle='--', label='Fit Line')
        corr, pval = stats.pearsonr(angles, f1_scores)
        plt.title(f'Correlation between Cue Weight Angle and F1 Score - {x_modality}\n r={corr:.2f}, p={pval:.3f}')
        plt.xlabel('Angle between Cue 1 and Cue 2 Weights (degrees)')
        plt.ylabel('Mean F1 Score')
        plt.ylim(.1,.9)
        plt.tight_layout()
        plt.legend()
        plt.show()

In [ ]:
# fetch the data again, this time unfiltered

svm_results = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
svm_results


In [ ]:
# fetch data again, this time the t0 events

t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events.index = t0_events.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
t0_events.set_index("trial_id", append=True, inplace=True)
t0_events

In [ ]:
# hardcoded 2 std boundaries of 2 random angles within a high dim space, matching dims of our data:
# so HP eg is 20 dims and is therfore more spread out around 90, versus mPFC with eg 54 dims
# calculcated later, but placed here for convenience

mPFC_upper = 106.81
mPFC_lower = 72.81
HP_upper = 123.52
HP_lower = 56.06
HP_mPFC_upper = 105.00
HP_mPFC_lower = 74.63
# insetad do dict
angle_2std_thr = {
    'mPFC': (mPFC_lower, mPFC_upper),
    'HP': (HP_lower, HP_upper),
    'HP+mPFC': (HP_mPFC_lower, HP_mPFC_upper)
}

In [ ]:
# the numbers above come from here

plt.close("all")

# svm_res
for x_modal in ['mPFC', 'HP', 'HP+mPFC']:
    plt.figure(figsize=(5,5))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                            (svm_results['predict_y_name'] == 'cue')
                        ].loc[:8]
    
    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    # convert to degrees
    degrees = (np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)).flatten()
    
    hist = plt.hist(degrees, bins=80, color='gray')
    # draw in 2 std dev lines
    mean_angle = np.nanmean(degrees)
    std_angle = np.nanstd(degrees)
    print(mean_angle + 2*std_angle, x_modal)
    print(mean_angle - 2*std_angle, x_modal)
    plt.axvline(mean_angle + 2*std_angle, color='lightblue', linestyle='--', label=f'{mean_angle + 2*std_angle:.2f}°')
    plt.axvline(mean_angle - 2*std_angle, color='lightblue', linestyle='--', label=f'{mean_angle - 2*std_angle:.2f}°')
    plt.title(f'Histogram of Cosine Similarity of SVM Weights - {x_modal}')
    plt.legend()
    plt.xlabel('Angle (degrees)') 
    plt.ylabel('Count')
    plt.tight_layout()
plt.show()

In [ ]:
# POSTER PLOTS

def get_session_data(s_id, t0_events, behavior, svm_results, x_modality):
    """Get all relevant data for a single session and modality"""
    # Get SVM results
    X_modality_svm_results = svm_results.loc[
        (svm_results.index.get_level_values('session_id') == s_id) & 
        (svm_results['X_modality'] == x_modality) &
        (svm_results['predict_y_name'] == y_predicted_name) &
        (svm_results['interval_name'].isin(which_intervals))
    ]
    
    # Get behavior trajectories
    all_pos = {}
    print("\nGetting positions data for session:", s_id)
    for which_interval in which_intervals:
        print("Interval ", which_interval)
        # all_pos[which_interval] = []
        trial_poss = []
        for trial_id, full_intrvl in t0_events.loc[s_id, which_interval].dropna().items():

            trial_behavior = behavior.xs(s_id, level='session_id')
            trial_behavior = trial_behavior[trial_behavior['trial_id'] == trial_id]
            dat = trial_behavior.loc[trial_behavior.index.overlaps(full_intrvl)]
            print(dat.shape, end="; ")
            trial_poss.append(dat.frame_position.values)

        # Filter trials with wrong length
        lengths = pd.Series([len(x) for x in trial_poss])
        correct_length = lengths.value_counts().index[0]
        print(f"\n{correct_length=}")
        trial_pos = [pos for pos in trial_poss if len(pos) == correct_length]
        all_pos[which_interval] = np.stack(trial_pos)
        print("SVM data shape: ", X_modality_svm_results[X_modality_svm_results.interval_name == which_interval].shape)
        print()
    return X_modality_svm_results, all_pos, dat.index.mid.values - dat.index.mid[0]

def create_subplot(axes, all_pos, svm_res, title, y_predicted_name):
    """Create a single subplot with trajectories and F1 score coloring"""
    
    # iter intervals
    offset = 0
    xticks = []
    for which_interval in which_intervals:
        # Plot individual trajectories
        this_interval_pos = all_pos[which_interval]
        svm_res_interval = svm_res[svm_res.interval_name == which_interval]

        for pos in this_interval_pos:
            axes[0].plot(svm_res_interval.timebin + offset, pos[1:-1], alpha=0.05, color='k', zorder=4)

        # Plot bin boundaries
        for t in svm_res_interval.timebin:
            axes[0].axvline(x=t-.5 + offset, color='k', linestyle='--', alpha=0.04)

        x_0 = 3+ offset if which_interval == 'cue_entry_interval' else 10+ offset
        xticks.extend([x_0, x_0+25])
        axes[0].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[1].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[2].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[2].axhline(y=x_0, color='k', linestyle='--', alpha=0.4)
        
        scatter = axes[0].scatter(svm_res_interval.timebin + offset, (this_interval_pos).mean(axis=0)[1:-1],
                            c=svm_res_interval.f1_mean,
                            cmap=plt.cm.viridis,
                            norm=Normalize(vmin=0.2, vmax=0.8),
                            alpha=1,
                            s=30,
                            zorder=5)
        
        # plot performance over timepoints
        axes[1].plot(svm_res_interval.timebin + offset, svm_res_interval.f1_mean, color='black', alpha=0.6, zorder=5)
        # draw confidence intervals svm_res.f1_ci_lower, svm_res.f1_ci_upper
        axes[1].fill_between(svm_res_interval.timebin + offset, svm_res_interval.f1_ci_lower, 
                             svm_res_interval.f1_ci_upper, color='black', alpha=0.07,
                             edgecolor=None)
        
        axes[2].axvline(svm_res_interval.timebin.max()+offset +.5, color='white', linewidth=3, alpha=1, zorder=5)
        axes[2].axhline(svm_res_interval.timebin.max()+offset +.5, color='white', linewidth=3, alpha=1, zorder=5)
        offset += svm_res_interval.timebin.max() +1
    
    axes[1].axhline(0.5, color='black', linestyle=':', label='Chance Level')
    axes[1].set_ylabel(f'Decode {y_predicted_name} F1 Score')
    # y grid
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.5)
    axes[1].set_ylim(.1,.9)
    info_str = (f"$n_{{y=0}}={svm_res_interval.n_negatives.values[0]}$, "
                f"$n_{{y=1}}={svm_res_interval.n_positives.values[0]}$")
                # f"n bootstr.: {svm_res_interval.n_iterations.values[0]} ") #, failed: {svm_res_interval.n_success_iterations.values[0]-svm_res_interval.n_iterations.values[0]}, ")
    axes[1].text(0.95, 0.75, info_str, transform=axes[1].transAxes,
        fontsize=10, verticalalignment='bottom', horizontalalignment='right')
        
        
            
    # Add reference lines
    # rectangle for cue zone
    # axes[0].hlines(y=-120, xmin=0, xmax=offset, color='gray', linestyle='--', label='Cue Zone Visible')
    axes[0].fill_betweenx(y=[-80, 0], x1=0, x2=offset, color='black', hatch='/', alpha=0.3, label='Cue Zone')
    axes[0].fill_betweenx(y=[50, 130], x1=0, x2=offset, color='darkgray', alpha=0.6, label='R1 Zone')
    axes[0].fill_betweenx(y=[170, 250], x1=0, x2=offset, color='lightgray', alpha=0.6, label='R2 Zone')

    # Labels
    # axes[0].legend()
    axes[0].set_title(title)
    axes[0].tick_params(labelleft=False, labelbottom=False)

    return scatter, xticks

def plot_session_analysis(s_id, t0_events, behavior, svm_results):
    """Plot complete analysis for one session"""
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(7, 6), height_ratios=[1.5, 1, 2], 
                             sharex=True)
    # reduce space between subplots
    plt.subplots_adjust(hspace=0.01, wspace=0.01)
    # y_predicted_name = svm_results['predict_y_name'].values[0] # all the same
    # y_predicted_name = 'cue'
    
    # Plot first row with existing analysis
    for i, x_modality in enumerate(x_modalities):
    # for i, x_modality in enumerate(['mPFC']):
        svm_res, all_pos, all_times = get_session_data(s_id, t0_events, behavior, svm_results, x_modality)
        # f1_scores = [np.nan, *svm_res.f1_mean.to_list(), np.nan]
        scatter, xticks = create_subplot(axes[:,i], all_pos, svm_res, 
                                 f'{y_predicted_name} decoding from \n{x_modality}', y_predicted_name)
        
        
        # Get the timebin values for x-axis alignment
        timebins = np.arange(len(svm_res))
        # Plot heatmap with aligned x-axis
        Ws = np.array(svm_res.w.to_list())
        cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
        cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)

        high_angles_alpha = ((cos_sim_matrix > angle_2std_thr[x_modality][1]) | (cos_sim_matrix < angle_2std_thr[x_modality][0])).astype(float)
        high_f1_mask = svm_res.f1_mean.values > .55
        high_angles_alpha[high_angles_alpha==False] = .3
        # high_angles_alpha[~high_f1_mask,:] = .3
        # high_angles_alpha[:, ~high_f1_mask] = .3

        # Use pcolormesh instead of imshow for better alignment
        X, Y = np.meshgrid(timebins, timebins)
        im = axes[2,i].pcolormesh(X, Y, cos_sim_matrix, 
                                  alpha=high_angles_alpha,
                                 cmap='bwr_r', vmin=0, vmax=180)
        
        axes[2,i].set_xticks(xticks)
        axes[2,i].set_xticklabels(['t=0' if i%2==0 else '1s' for i in range(len(xticks))])
    
        # Set proper aspect ratio and limits
        # axes[2,i].set_aspect('equal')
        axes[2,i].set_xlim(-0.5, len(timebins)-0.5)
        axes[2,i].set_ylim(-0.5, len(timebins)-0.5)
        axes[2,i].tick_params(labelleft=False)
        # turn off all spines
        [sp.set_visible(False) for sp in axes[0,i].spines.values()]
        [sp.set_visible(False) for sp in axes[1,i].spines.values()]
        [sp.set_visible(False) for sp in axes[2,i].spines.values()]
        
        
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=axes[1,-1])
    cbar.set_ticks([.75, .5, .25])
    cbar.set_label('SVM F1 Score', labelpad=15)
    
    # other colorbar
    cbar2 = plt.colorbar(im, ax=axes[-1,-1])
    cbar2.set_ticks([0, 60, 90, 120, 180])
    cbar2.set_label('Angle between SVM planes (degrees)', labelpad=15)

    plt.suptitle(f'Session {s_id}', fontsize=16, y = .99, )
    plt.tight_layout()
    plt.show()

    fname = os.path.join(output_dir, f'session_S{s_id}_{y_predicted_name}_svm_analysis_S15_Rest.svg')
    fig.savefig(fname, dpi=300)
    #degree sign
    # 45 °


plt.close("all")
y_predicted_name = 'cue'
x_modalities = ['HP+mPFC', 'mPFC', 'HP']
# which_intervals = ['cue_entry_interval', ]'cue_exit_interval', 'R1_entry_interval', 'R2_entry_interval']
which_intervals = ['cue_entry_interval', 'R2_entry_interval',]

# Main execution
for s_id in svm_results.index.unique('session_id'):
    print(f"Plotting session {s_id}")
    # if s_id in (24,25):
    # if s_id not in (9,10,11,12,13,14):
        # session with cropped SS, skip
    if s_id not in (15,):
        
        continue
    plot_session_analysis(s_id, t0_events, behavior, svm_results)
    # if s_id == 3:

    # break
    
    
# First proper plot using in poster. Play around with the part directly above
# to select sessions, intervals, y_predicted_name, x_modalities

In [ ]:
# outlier plot: the overlap of different labels in session 14


trialw_behavior = analytics.get_analytics('BehaviorTrialwise', session_names=session_names[14:15])
trialw_behavior.index = trialw_behavior.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
trialw_behavior.set_index("trial_id", append=True, inplace=True)
trialw_behavior.loc[:, 'trial_outcome'] = trialw_behavior.loc[:,'trial_outcome'].clip(0,1)

print(trialw_behavior.cue.value_counts())
print(trialw_behavior.choice_R1.value_counts())
print(trialw_behavior.trial_outcome.value_counts())

# Create visualization of label overlap
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
plt.subplots_adjust(hspace=0.05)  # Reduce space between plots

# Get data
cue_1_trials = trialw_behavior[trialw_behavior.cue == 1]
cue_2_trials = trialw_behavior[trialw_behavior.cue == 2]

# Top plot: Cue distribution (Orange for Cue 1, Purple for Cue 2)
cue_counts = [len(cue_1_trials), len(cue_2_trials)]
axes[0].barh([0], [cue_counts[0]], left=0, color='orange', label='Cue 1', height=0.6)
axes[0].barh([0], [cue_counts[1]], left=cue_counts[0], color='purple', label='Cue 2', height=0.6)
axes[0].set_ylim(-0.5, 0.5)
axes[0].set_xlim(0, sum(cue_counts))
axes[0].set_yticks([0])
axes[0].set_yticklabels(['Cue'])
# axes[0].legend(loc='upper right')
axes[0].set_title('Distribution of Labels Across Trials', fontsize=14, fontweight='bold')
axes[0].text(cue_counts[0]/2, 0, f'n={cue_counts[0]}', ha='center', va='center', fontsize=10, fontweight='bold')
axes[0].text(cue_counts[0] + cue_counts[1]/2, 0, f'n={cue_counts[1]}', ha='center', va='center', fontsize=10, fontweight='bold')

# Middle plot: Choice_R1 distribution split by cue (Dark gray for Stop R1, Gray for Skip R1)
cue1_choice_counts = [
    len(cue_1_trials[cue_1_trials.choice_R1 == False]),    # Skip R1
    len(cue_1_trials[cue_1_trials.choice_R1 == True])  # Stop R1
]
print(cue_1_trials)
print(cue1_choice_counts)
cue2_choice_counts = [
    len(cue_2_trials[cue_2_trials.choice_R1 == False]),    # Skip R1
    len(cue_2_trials[cue_2_trials.choice_R1 == True]),  # Stop R1
]

# Cue 1 side
axes[1].barh([0], [cue1_choice_counts[0]], left=0, color='#404040', label='Stop R1', height=0.6)
axes[1].barh([0], [cue1_choice_counts[1]], left=cue1_choice_counts[0], color='#808080', label='Skip R1', height=0.6)
# Cue 2 side
axes[1].barh([0], [cue2_choice_counts[0]], left=cue_counts[0], color='#404040', height=0.6)
axes[1].barh([0], [cue2_choice_counts[1]], left=cue_counts[0] + cue2_choice_counts[0], color='#808080', height=0.6)

axes[1].set_ylim(-0.5, 0.5)
axes[1].set_xlim(0, sum(cue_counts))
axes[1].set_yticks([0])
axes[1].set_yticklabels(['Choice R1'])
# axes[1].legend(loc='upper right')
axes[1].axvline(cue_counts[0], color='black', linestyle='--', alpha=0.3)

# Add counts
axes[1].text(cue1_choice_counts[0]/2, 0, f'{cue1_choice_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[1].text(cue1_choice_counts[0] + cue1_choice_counts[1]/2, 0, f'{cue1_choice_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')
axes[1].text(cue_counts[0] + cue2_choice_counts[0]/2, 0, f'{cue2_choice_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[1].text(cue_counts[0] + cue2_choice_counts[0] + cue2_choice_counts[1]/2, 0, f'{cue2_choice_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')

# Bottom plot: Trial outcome distribution split by cue (Red for no reward, Green for reward)
cue1_outcome_counts = [
    len(cue_1_trials[cue_1_trials.trial_outcome == 0]),  # No Reward
    len(cue_1_trials[cue_1_trials.trial_outcome == 1])   # Reward
]
cue2_outcome_counts = [
    len(cue_2_trials[cue_2_trials.trial_outcome == 0]),  # No Reward
    len(cue_2_trials[cue_2_trials.trial_outcome == 1])   # Reward
]

# Cue 1 side
axes[2].barh([0], [cue1_outcome_counts[0]], left=0, color='red', label='No Reward', height=0.6)
axes[2].barh([0], [cue1_outcome_counts[1]], left=cue1_outcome_counts[0], color='green', label='Reward', height=0.6)
# Cue 2 side
axes[2].barh([0], [cue2_outcome_counts[0]], left=cue_counts[0], color='red', height=0.6)
axes[2].barh([0], [cue2_outcome_counts[1]], left=cue_counts[0] + cue2_outcome_counts[0], color='green', height=0.6)

axes[2].set_ylim(-0.5, 0.5)
axes[2].set_xlim(0, sum(cue_counts))
axes[2].set_yticks([0])
axes[2].set_yticklabels(['Outcome'])
axes[2].legend(loc='upper right')
axes[2].axvline(cue_counts[0], color='black', linestyle='--', alpha=0.3)
axes[2].set_xlabel('Number of Trials', fontsize=12)

# Add counts
axes[2].text(cue1_outcome_counts[0]/2, 0, f'{cue1_outcome_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[2].text(cue1_outcome_counts[0] + cue1_outcome_counts[1]/2, 0, f'{cue1_outcome_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')
axes[2].text(cue_counts[0] + cue2_outcome_counts[0]/2, 0, f'{cue2_outcome_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[2].text(cue_counts[0] + cue2_outcome_counts[0] + cue2_outcome_counts[1]/2, 0, f'{cue2_outcome_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')


plt.tight_layout()
plt.show()
fname = os.path.join(output_dir, f'trial_label_distribution.svg')
fig.savefig(fname, dpi=300)

In [ ]:
# plot for showing that decoding performance goes up over sessions


# y_predicted_name = 'cue'
y_predicted_name = 'choice_R1'
which_intervals = ['cue_entry_interval', 'R1_entry_interval',]
x_modal = 'mPFC'
fname = f'avg_f1_vs_session_{y_predicted_name}_{x_modal}.svg'
# s_ids = [9,10,11,12,13,14]
s_ids = [6,7,9,10,11,12,13,14]

avg_f1 = []
for s_id in svm_results.index.unique('session_id'):
    if s_id not in s_ids:
        continue
    s_svm_results = svm_results.xs(s_id, level='session_id')
    s_svm_results = s_svm_results[(s_svm_results['predict_y_name'] == y_predicted_name) &
                              (s_svm_results['X_modality'] == x_modal) &
                              (s_svm_results['interval_name'].isin(which_intervals))]
    avg_f1.append((s_id, s_svm_results.f1_mean.mean()))

avg_f1 = np.array(avg_f1)

plt.figure(figsize=(4,3))
# Color points by F1 score using viridis colormap scaled from 0.2 to 0.8
scatter = plt.scatter(avg_f1[:,0], avg_f1[:,1], 
                     c=avg_f1[:,1],
                     cmap='viridis',
                     norm=Normalize(vmin=0.2, vmax=0.8),
                     s=100,
                     edgecolors='black',
                     linewidths=0.5)

# Linear fit
m, b = np.polyfit(avg_f1[:,0], avg_f1[:,1], 1)
plt.plot(avg_f1[:,0], m*avg_f1[:,0] + b, color='gray', linestyle='--', linewidth=2)
plt.title(y_predicted_name)

# Correlation
corr, pval = stats.pearsonr(avg_f1[:,0], avg_f1[:,1])
plt.text(0.95, 0.1, f'r={corr:.2f}\np={pval:.3f} *', transform=plt.gca().transAxes,
         verticalalignment='bottom', horizontalalignment='right')

plt.xlabel('Session ID')
plt.ylabel('Average F1 Score')
plt.ylim(.2,.8)
plt.yticks([.25, .5, .6, .75])
plt.gca().yaxis.grid(True, linestyle='--', alpha=0.5)
plt.xticks(s_ids)
plt.xlim(min(s_ids)-.5, max(s_ids)+.5)
plt.xlabel('Session')
plt.axhline(.5, color='black', linestyle=':')
# no spines except left
[sp.set_visible(False) for sp in plt.gca().spines.values() if sp != plt.gca().spines['left']]

# # Add colorbar
# cbar = plt.colorbar(scatter)
# cbar.set_ticks([0.25, 0.5, 0.75])
# cbar.set_label('F1 Score', labelpad=15)

plt.tight_layout()
plt.show()

plt.savefig(os.path.join(output_dir, fname), dpi=300)

In [ ]:
# angles between SVM planes across sessions for the same y_prediction

from scipy.ndimage import gaussian_filter

plt.close("all")
y_predicted_name = 'choice_R1'
which_intervals = ['cue_entry_interval', 'R1_entry_interval']
session_subset = [6,7,9,10,11,12,13,14]

for x_modal in ['mPFC', 'HP+mPFC']:
    plt.figure(figsize=(20,20))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                          (svm_results['predict_y_name'] == y_predicted_name) &
                          (svm_results['interval_name'].isin(which_intervals))
                          ].set_index('timebin', append=True)
    svm_res.index = svm_res.index.droplevel(("paradigm_id", "animal_id", "entry_id"))
    svm_res = svm_res.loc[svm_res.index.get_level_values('session_id').isin(session_subset)]

    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)

    # Apply Gaussian blur
    cos_sim_matrix_blurred = gaussian_filter(cos_sim_matrix, sigma=0)

    high_angles_alpha = ((cos_sim_matrix_blurred > angle_2std_thr[x_modal][1]) | 
                         (cos_sim_matrix_blurred < angle_2std_thr[x_modal][0])).astype(float)
    high_f1 = svm_res.f1_mean.values > .55
    high_angles_alpha[~high_f1,:] = .3
    high_angles_alpha[:, ~high_f1] = .3
    high_angles_alpha[high_angles_alpha==False] = .3

    # Use pcolormesh instead of imshow for better alignment
    X, Y = np.meshgrid(np.arange(cos_sim_matrix.shape[1]), np.arange(cos_sim_matrix.shape[0]))
    im = plt.pcolormesh(X, Y, cos_sim_matrix_blurred, 
                                alpha=high_angles_alpha,
                                cmap='bwr_r', vmin=0, vmax=180)
    
    plt.colorbar(im, label='Angle (degrees)')
    plt.title(f'Cosine Similarity of SVM Weights - {x_modal} - {y_predicted_name}')
    plt.xlabel('Session-Timebin Index') 
    plt.ylabel('Session-Timebin Index')
    
    # draw every session-timebin tick where timebin == 20 (middle of cue visible period)
    ticks = [t if t[1] == 20 else None for t in svm_res.index]
    new_s_ids = np.diff(svm_res.index.get_level_values('session_id')) != 0
    new_s_ids = np.insert(new_s_ids, 0, True)
    new_s_ids = np.arange(len(svm_res))[new_s_ids]
    
    # draw axh and v lines
    for t in range(len(svm_res)):
        if ticks[t] is not None:
            plt.axhline(y=t-20, color='black', linestyle='--', linewidth=.5, alpha=0.1)
            plt.axvline(x=t-20, color='black', linestyle='--', linewidth=.5, alpha=0.1)
    all_lbls = []
    for ns in new_s_ids:
        lw = 2 if svm_res.index[ns][0] in (20,26) else 0.5
        plt.axhline(y=ns-0.5, color='gray', linestyle='-', linewidth=lw, alpha=0.2)
        plt.axvline(x=ns-0.5, color='gray', linestyle='-', linewidth=lw, alpha=0.2)
        lbl = f"S{svm_res.index[ns][0]:02d}"
        plt.text(ns, -5, lbl, 
                 verticalalignment='bottom', horizontalalignment='left', fontsize=6)
        plt.text(-5, ns+5, lbl,
                 verticalalignment='top', horizontalalignment='right', fontsize=6)
        all_lbls.append(lbl)
        
    plt.yticks(new_s_ids, all_lbls)
    plt.xticks(new_s_ids, all_lbls)
    plt.tick_params(axis='both', which='both', labelleft=False, labelright=True, right=True)
    plt.tight_layout()
    plt.show()
    
    # save figure
    fname = os.path.join(output_dir, f'cosine_similarity_{y_predicted_name}_{x_modal}_blurred.png')
    plt.savefig(fname, dpi=300)
    # break
    
    plt.figure(figsize=(6,6))
    print(f"shape: {cos_sim_matrix.shape}")
    high_f1_data = cos_sim_matrix[high_f1][:, high_f1].flatten()
    print(f"n high F1: {len(high_f1_data)}")
    low_f1_data = cos_sim_matrix[~high_f1][:, ~high_f1].flatten()
    print(f"n low F1: {len(low_f1_data)}")
    
    plt.hist(high_f1_data[high_f1_data>2], bins=50, color='green', edgecolor='black', alpha=0.5 , density=True)
    plt.hist(low_f1_data[low_f1_data>2], bins=50, color='gray', edgecolor='black', alpha=0.5   , density=True)
    # t test
    t_stat, p_val = stats.ttest_ind(high_f1_data, low_f1_data, equal_var=False)
    plt.text(0.95, 0.95, f't={t_stat:.2f}\np={p_val:.3e}', transform=plt.gca().transAxes,
                verticalalignment='top', horizontalalignment='right')
    plt.title(f'high F1 - Cosine Similarity Angles - {x_modal} - {y_predicted_name}')
    plt.xlabel('Angle (degrees)')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# nasty gpt slop: THis is the panel 8, copmaring angles between predicters and sessions:

from scipy.ndimage import gaussian_filter
from scipy import stats

plt.close("all")
y_predicted_names = ['cue', 'choice_R1']
which_intervals = ['cue_entry_interval', 'R1_entry_interval']
session_subset = [9, 11, 12, 13, 14]
thr = .55

low_f1_col = '#445e8aff'
high_f1_col = '#9bd15aff'

def plot_cross_predictor_comparison(x_modal, y_predicted_names, which_intervals, session_subset, thr, svm_results):
    """Plot cross-predictor comparison for a single brain region"""
    # Create figure with 6 columns and 2 rows
    fig, axes = plt.subplots(figsize=(15, 8), ncols=6, nrows=2, 
                            gridspec_kw={'height_ratios': [6, 1.5]})
    
    # Get SVM results
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                          (svm_results['predict_y_name'].isin(y_predicted_names)) &
                          (svm_results['interval_name'].isin(which_intervals))
                          ].set_index('timebin', append=True)
    svm_res.index = svm_res.index.droplevel(("paradigm_id", "animal_id", "entry_id"))
    svm_res = svm_res.loc[svm_res.index.get_level_values('session_id').isin(session_subset)]

    # Initialize arrays for storing results
    cue_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cue_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    choice_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    choice_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    
    # Also store std for error bars
    cue_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cue_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    choice_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    choice_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    
    # Store raw angle distributions for statistical testing
    cue_high_f1_angles_raw = {}
    cue_low_f1_angles_raw = {}
    choice_high_f1_angles_raw = {}
    choice_low_f1_angles_raw = {}
    cross_predictor_high_f1_angles_raw = {}
    cross_predictor_low_f1_angles_raw = {}
    
    # Compute within-predictor similarities for cue
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            s_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id, level='session_id')
            s2_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id_2, level='session_id')
            
            s_Ws = np.array(s_svm_res.w.to_list())
            s2_Ws = np.array(s2_svm_res.w.to_list())
            
            # Compute cosine similarity
            cos_sim_matrix = np.dot(s_Ws, s2_Ws.T) / (np.linalg.norm(s_Ws, axis=1)[:, np.newaxis] * 
                                                       np.linalg.norm(s2_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # High F1 mask
            s_high_f1 = s_svm_res.f1_mean.values > thr
            s2_high_f1 = s2_svm_res.f1_mean.values > thr
            
            high_f1_angles = cos_sim_matrix[np.ix_(s_high_f1, s2_high_f1)]
            low_f1_angles = cos_sim_matrix[np.ix_(~s_high_f1, ~s2_high_f1)]
            
            cue_high_f1_angles[i, j] = np.mean(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            cue_low_f1_angles[i, j] = np.mean(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            cue_high_f1_angles_std[i, j] = np.std(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            cue_low_f1_angles_std[i, j] = np.std(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                cue_high_f1_angles_raw[i] = high_f1_angles.flatten() if high_f1_angles.size > 0 else np.array([])
                cue_low_f1_angles_raw[i] = low_f1_angles.flatten() if low_f1_angles.size > 0 else np.array([])
    
    # Compute within-predictor similarities for choice_R1
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            s_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id, level='session_id')
            s2_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id_2, level='session_id')
            
            s_Ws = np.array(s_svm_res.w.to_list())
            s2_Ws = np.array(s2_svm_res.w.to_list())
            
            # Compute cosine similarity
            cos_sim_matrix = np.dot(s_Ws, s2_Ws.T) / (np.linalg.norm(s_Ws, axis=1)[:, np.newaxis] * 
                                                       np.linalg.norm(s2_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # High F1 mask
            s_high_f1 = s_svm_res.f1_mean.values > thr
            s2_high_f1 = s2_svm_res.f1_mean.values > thr
            
            high_f1_angles = cos_sim_matrix[np.ix_(s_high_f1, s2_high_f1)]
            low_f1_angles = cos_sim_matrix[np.ix_(~s_high_f1, ~s2_high_f1)]
            
            choice_high_f1_angles[i, j] = np.mean(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            choice_low_f1_angles[i, j] = np.mean(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            choice_high_f1_angles_std[i, j] = np.std(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            choice_low_f1_angles_std[i, j] = np.std(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                choice_high_f1_angles_raw[i] = high_f1_angles.flatten() if high_f1_angles.size > 0 else np.array([])
                choice_low_f1_angles_raw[i] = low_f1_angles.flatten() if low_f1_angles.size > 0 else np.array([])
    
    # Compute cross-predictor similarities (cue vs choice_R1)
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            cue_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id, level='session_id')
            choice_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id_2, level='session_id')
            
            cue_Ws = np.array(cue_svm_res.w.to_list())
            choice_Ws = np.array(choice_svm_res.w.to_list())
            
            # Compute cross-predictor angles
            cos_sim_matrix = np.dot(cue_Ws, choice_Ws.T) / (np.linalg.norm(cue_Ws, axis=1)[:, np.newaxis] * 
                                                             np.linalg.norm(choice_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # Flip around 90 degrees because cue 0 and choice 1, alignment is blue
            cos_sim_matrix = 180 - cos_sim_matrix
            
            # Masks for high and low F1
            cue_high_f1 = cue_svm_res.f1_mean.values > thr
            choice_high_f1 = choice_svm_res.f1_mean.values > thr
            
            # High F1 cross-predictor angles
            cross_high_angles = cos_sim_matrix[np.ix_(cue_high_f1, choice_high_f1)]
            cross_predictor_high_f1_angles[i, j] = np.mean(cross_high_angles) if cross_high_angles.size > 0 else np.nan
            cross_predictor_high_f1_angles_std[i, j] = np.std(cross_high_angles) if cross_high_angles.size > 0 else np.nan
            
            # Low F1 cross-predictor angles
            cross_low_angles = cos_sim_matrix[np.ix_(~cue_high_f1, ~choice_high_f1)]
            cross_predictor_low_f1_angles[i, j] = np.mean(cross_low_angles) if cross_low_angles.size > 0 else np.nan
            cross_predictor_low_f1_angles_std[i, j] = np.std(cross_low_angles) if cross_low_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                cross_predictor_high_f1_angles_raw[i] = cross_high_angles.flatten() if cross_high_angles.size > 0 else np.array([])
                cross_predictor_low_f1_angles_raw[i] = cross_low_angles.flatten() if cross_low_angles.size > 0 else np.array([])

    # Plot all six heatmaps (top row)
    im0 = axes[0, 0].imshow(cue_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 0].set_title(f'Cue Within-Predictor\n(High F1 > {thr})\n{x_modal}')
    axes[0, 0].set_xlabel('Session ID')
    axes[0, 0].set_ylabel('Session ID')
    axes[0, 0].set_xticks(range(len(session_subset)))
    axes[0, 0].set_xticklabels(session_subset)
    axes[0, 0].set_yticks(range(len(session_subset)))
    axes[0, 0].set_yticklabels([f'S{s}' for s in session_subset])
    
    im1 = axes[0, 1].imshow(cue_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 1].set_title(f'Cue Within-Predictor\n(Low F1 ≤ {thr})\n{x_modal}')
    axes[0, 1].set_xlabel('Session ID')
    axes[0, 1].set_ylabel('Session ID')
    axes[0, 1].set_xticks(range(len(session_subset)))
    axes[0, 1].set_xticklabels(session_subset)
    axes[0, 1].set_yticks(range(len(session_subset)))
    axes[0, 1].set_yticklabels([f'S{s}' for s in session_subset])
    
    im2 = axes[0, 2].imshow(choice_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 2].set_title(f'Choice_R1 Within-Predictor\n(High F1 > {thr})\n{x_modal}')
    axes[0, 2].set_xlabel('Session ID')
    axes[0, 2].set_ylabel('Session ID')
    axes[0, 2].set_xticks(range(len(session_subset)))
    axes[0, 2].set_xticklabels(session_subset)
    axes[0, 2].set_yticks(range(len(session_subset)))
    axes[0, 2].set_yticklabels([f'S{s}' for s in session_subset])
    
    im3 = axes[0, 3].imshow(choice_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 3].set_title(f'Choice_R1 Within-Predictor\n(Low F1 ≤ {thr})\n{x_modal}')
    axes[0, 3].set_xlabel('Session ID')
    axes[0, 3].set_ylabel('Session ID')
    axes[0, 3].set_xticks(range(len(session_subset)))
    axes[0, 3].set_xticklabels(session_subset)
    axes[0, 3].set_yticks(range(len(session_subset)))
    axes[0, 3].set_yticklabels([f'S{s}' for s in session_subset])
    
    im4 = axes[0, 4].imshow(cross_predictor_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 4].set_title(f'Cross-Predictor Angles\n(Cue vs Choice_R1, High F1)\n{x_modal}')
    axes[0, 4].set_xlabel('Session ID (Choice_R1)')
    axes[0, 4].set_ylabel('Session ID (Cue)')
    axes[0, 4].set_xticks(range(len(session_subset)))
    axes[0, 4].set_xticklabels(session_subset)
    axes[0, 4].set_yticks(range(len(session_subset)))
    axes[0, 4].set_yticklabels([f'S{s}' for s in session_subset])
    
    im5 = axes[0, 5].imshow(cross_predictor_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 5].set_title(f'Cross-Predictor Angles\n(Cue vs Choice_R1, Low F1)\n{x_modal}')
    axes[0, 5].set_xlabel('Session ID (Choice_R1)')
    axes[0, 5].set_ylabel('Session ID (Cue)')
    axes[0, 5].set_xticks(range(len(session_subset)))
    axes[0, 5].set_xticklabels(session_subset)
    axes[0, 5].set_yticks(range(len(session_subset)))
    axes[0, 5].set_yticklabels([f'S{s}' for s in session_subset])
    
    # Plot line plots (bottom row) - ROTATED with sessions on y-axis
    y_sessions = range(len(session_subset))
    
    # Helper function to add significance stars
    def add_significance_stars(ax, y_sessions, high_angles_raw, low_angles_raw, x_pos):
        for i in y_sessions:
            if i in high_angles_raw and i in low_angles_raw:
                high_data = high_angles_raw[i]
                low_data = low_angles_raw[i]
                if len(high_data) > 1 and len(low_data) > 1:
                    _, p_val = stats.ttest_ind(high_data, low_data, equal_var=False)
                    if p_val < (0.05/len(y_sessions)):  # Bonferroni correction
                        ax.text(x_pos, i, '*', ha='left', va='center', fontsize=16, fontweight='bold')
    
    # Plot 0: Cue - both high and low F1 (ROTATED)
    axes[1, 0].plot(cue_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 0].errorbar(cue_high_f1_angles[:, -1], y_sessions, xerr=cue_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 0].plot(cue_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 0].errorbar(cue_low_f1_angles[:, -1], y_sessions, xerr=cue_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 0], y_sessions, cue_high_f1_angles_raw, cue_low_f1_angles_raw, 108)
    axes[1, 0].set_xlabel('Angle (°)')
    axes[1, 0].set_ylabel('Session ID')
    axes[1, 0].set_yticks(y_sessions)
    axes[1, 0].set_yticklabels([s for s in session_subset])
    axes[1, 0].set_xlim(70, 100)
    axes[1, 0].set_xticks([80,90])
    axes[1, 0].axvline(90, color='gray', linestyle='--', alpha=0.5)
    axes[1, 0].grid(True, alpha=0.3)
    # axes[1, 0].legend()
    axes[1, 0].spines['top'].set_visible(False)
    axes[1, 0].spines['right'].set_visible(False)
    axes[1, 0].spines['bottom'].set_visible(False)
    
    # Hide plot 1
    axes[1, 1].axis('off')
    
    # Plot 2: Choice - both high and low F1 (ROTATED)
    axes[1, 2].plot(choice_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 2].errorbar(choice_high_f1_angles[:, -1], y_sessions, xerr=choice_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 2].plot(choice_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 2].errorbar(choice_low_f1_angles[:, -1], y_sessions, xerr=choice_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 2], y_sessions, choice_high_f1_angles_raw, choice_low_f1_angles_raw, 108)
    axes[1, 2].set_xlabel('Angle (°)')
    axes[1, 2].set_ylabel('Session ID')
    axes[1, 2].set_yticks(y_sessions)
    axes[1, 2].set_yticklabels(session_subset)
    axes[1, 2].set_xlim(70, 100)
    axes[1, 2].set_xticks([80,90])
    axes[1, 2].axvline(90, color='gray', linestyle='--', alpha=0.5)
    axes[1, 2].grid(True, alpha=0.3)
    # axes[1, 2].legend()
    axes[1, 2].spines['top'].set_visible(False)
    axes[1, 2].spines['right'].set_visible(False)
    axes[1, 2].spines['bottom'].set_visible(False)
    
    # Hide plot 3
    axes[1, 3].axis('off')
    
    # Plot 4: Cross-predictor - both high and low F1 (ROTATED)
    axes[1, 4].plot(cross_predictor_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 4].errorbar(cross_predictor_high_f1_angles[:, -1], y_sessions, xerr=cross_predictor_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 4].plot(cross_predictor_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 4].errorbar(cross_predictor_low_f1_angles[:, -1], y_sessions, xerr=cross_predictor_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 4], y_sessions, cross_predictor_high_f1_angles_raw, cross_predictor_low_f1_angles_raw, 108)
    axes[1, 4].set_xlabel('Angle (°)')
    axes[1, 4].set_ylabel('Session ID')
    axes[1, 4].set_yticks(y_sessions)
    axes[1, 4].set_yticklabels(session_subset)
    axes[1, 4].set_xlim(70, 100)
    axes[1, 4].set_xticks([80,90])
    axes[1, 4].grid(True, alpha=0.3)
    # axes[1, 4].legend()
    axes[1, 4].spines['top'].set_visible(False)
    axes[1, 4].spines['right'].set_visible(False)
    axes[1, 4].spines['bottom'].set_visible(False)
    
    # Hide plot 5
    # axes[1, 5].axis('off')
    
    plt.tight_layout()
    
    # Save figure
    
    
    # return data
    return cross_predictor_high_f1_angles[:, -1]

# Call the function for each brain region
x = plot_cross_predictor_comparison('mPFC', y_predicted_names, which_intervals, session_subset, thr, svm_results)



s_ids = [9,11,12,13,14]
# these come froma plot above where we chec what propoertion of stop R1 trials coincide with cue=1 trials
print(x)
cov = [.64,.6,.8,.68,.75]
# x = np.arange(len(s_ids))
plt.scatter(x, cov, edgecolor=high_f1_col, s=100, color='gray', linewidths=1.5)
plt.ylim(.5,1)
plt.xticks((90,80))
plt.xlim(70, 100,)
plt.grid(linestyle='--', alpha=.5)
[sp.set_visible(False) for sp in plt.gca().spines.values()]

# Linear fit
m, b = np.polyfit(x, cov, 1)
plt.plot(x, m*x + b, color='gray', linestyle='--', linewidth=2)
# test correlation p value
corr, pval = stats.pearsonr(x, cov)
plt.title(f'r={corr:.2f}p={pval:.3f} Trial overlap', )
# tick right
plt.tick_params(axis='y', which='both', labelleft=False, labelright=True, right=True, left=False, )
plt.xlabel('Cross-Predictor Angle (degrees)')
plt.show()

fname = os.path.join(output_dir, f'cross_predictor-session_comparison_mPFC.svg')
print(fname)
plt.savefig(fname, dpi=300)

In [ ]:
# not sure what that one is anymore. I also counldn't find the cross predicter 
# heatmap plots (basciallcy the precurder of the one above)


plt.close("all")
which_intervals = ['cue_entry_interval', 'R1_entry_interval',] #'R2_entry_interval']
y_predicted_name = 'cue'

# svm_res
for x_modal in ['HP+mPFC', 'mPFC', 'HP']:
# for x_modal in ['mPFC', 'HP']:
    plt.figure(figsize=(20,20))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                            (svm_results['predict_y_name'] == y_predicted_name) &
                            (svm_results['interval_name'].isin(which_intervals))
                        ]
    
    svm_res.index = svm_res.set_index('timebin', append=True).index    
    # print(svm_res.index.value_counts().sort_index())
    
    svm_res = svm_res.loc[pd.IndexSlice[:,:,6:14]]
    print(svm_res)
    

    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    # convert to degrees
    cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
    alpha_mask = ((cos_sim_matrix > angle_2std_thr[x_modal][1]) | (cos_sim_matrix < angle_2std_thr[x_modal][0])).astype(float)
    alpha_mask[alpha_mask==False] = .3
    
    im = plt.imshow(cos_sim_matrix, cmap='bwr_r', vmin=0, vmax=180, alpha=alpha_mask)
    plt.colorbar(im, label='Angle (degrees)')
    plt.title(f'Cosine Similarity of SVM Weights - {x_modal} - {y_predicted_name} - Intervals: {which_intervals}')
    plt.xlabel('Session-Timebin Index') 
    plt.ylabel('Session-Timebin Index')
    # draw every session-timebin tick where timebin == 20 (middle of cue visible period)
    # ticks = [t if t[1] == 1 else None for t in svm_res.index]
    
    next_s_idx_mask = np.diff(svm_res.index.get_level_values('session_id'))
    print(next_s_idx_mask)
    next_s_idx_mask = np.insert(next_s_idx_mask, 0, 1).astype(bool)
    session_ticks = np.arange(len(svm_res))[next_s_idx_mask]
    plt.yticks(session_ticks, svm_res.index.unique('session_id') ,fontsize=6)
    plt.xticks(session_ticks, svm_res.index.unique('session_id') ,fontsize=6, rotation=90)
    # grid
    plt.grid(which='both', color='black', linestyle='--', alpha=0.4)
    
    # # draw axh and v lines
    # for t in range(len(svm_res)):
    #     if ticks[t] is not None:
    #         plt.axhline(y=t-20, color='black', linestyle='--', alpha=0.1)
    #         plt.axvline(x=t-20, color='black', linestyle='--', alpha=0.1)
    # plt.xticks(np.arange(len(svm_res)), ticks, fontsize=6)
    # plt.yticks(np.arange(len(svm_res)), ticks, fontsize=6)
    
    # tick labels are session_id
    # plt.yticks(np.arange(len(svm_res)), [f"S{t[0]:02d}" if t is not None else None for t in ticks], fontsize=6)
    # plt.xticks(np.arange(len(svm_res)), [f"S{t[0]:02d}" if t is not None else None for t in ticks], fontsize=6, rotation=90)
    
    plt.tight_layout()
    
    plt.show()
    fname = f"{output_dir}/cosine_similarity_{y_predicted_name}_{x_modal}.png"
    plt.savefig(fname, dpi=300)
    break